In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
sns.set_style("whitegrid")

II - PRE-TRAITEMENT

Lecture des csv

In [2]:
from pathlib import Path

BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data"

train = pd.read_csv(DATA_DIR / "train.csv",parse_dates=["Date"],dtype={"StateHoliday": str},low_memory=False)

test = pd.read_csv(DATA_DIR / "test.csv",parse_dates=["Date"],dtype={"StateHoliday": str},low_memory=False)

store = pd.read_csv(DATA_DIR / "store.csv",low_memory=False)

train = train.sort_values(["Store", "Date"]).reset_index(drop=True)
test  = test.sort_values(["Store", "Date"]).reset_index(drop=True)

# (Store, Date)
print("Shapes:", train.shape, test.shape, store.shape)
# Premiere lignes / colonnes
display(train.head(20))
# Types
display(train.dtypes)

Shapes: (1017209, 9) (41088, 8) (1115, 10)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,2,2013-01-01,0,0,0,0,a,1
1,1,3,2013-01-02,5530,668,1,0,0,1
2,1,4,2013-01-03,4327,578,1,0,0,1
3,1,5,2013-01-04,4486,619,1,0,0,1
4,1,6,2013-01-05,4997,635,1,0,0,1
5,1,7,2013-01-06,0,0,0,0,0,1
6,1,1,2013-01-07,7176,785,1,1,0,1
7,1,2,2013-01-08,5580,654,1,1,0,1
8,1,3,2013-01-09,5471,626,1,1,0,1
9,1,4,2013-01-10,4892,615,1,1,0,1


Store                     int64
DayOfWeek                 int64
Date             datetime64[us]
Sales                     int64
Customers                 int64
Open                      int64
Promo                     int64
StateHoliday                str
SchoolHoliday             int64
dtype: object

 Traitement des State Holidays
 la colonne contient '0' ou 'a','b','c' correspondant à
 0=no holiday, a=public, b=Easter, c=Christmas (par ex.).
 On crée un indicateur booléen et un code numérique pour le type.


In [3]:
# Normaliser pour éviter 0 vs '0'
train["StateHoliday"] = train["StateHoliday"].astype(str)

# Flag binaire: holiday ou non
train["StateHoliday_flag"] = (train["StateHoliday"] != "0").astype(int)

# Idem test
test["StateHoliday"] = test["StateHoliday"].astype(str)
test["StateHoliday_flag"] = (test["StateHoliday"] != "0").astype(int)

print("État des StateHoliday dans train:")
print(train["StateHoliday"].value_counts(dropna=False))
print(train[["StateHoliday", "StateHoliday_flag"]].head())


État des StateHoliday dans train:
StateHoliday
0    986159
a     20260
b      6690
c      4100
Name: count, dtype: int64
  StateHoliday  StateHoliday_flag
0            a                  1
1            0                  0
2            0                  0
3            0                  0
4            0                  0


Supprimer les jours où le magasin est fermé

Détecter les cas incohérences (open=1 & sales=0)

In [4]:
train = train[(train["Open"] == 1) & (train["Sales"] > 0)].copy()

Remplacer les NA numériques par la médiane

In [5]:
for col in train.select_dtypes(include=["int64", "float64"]).columns:
    train[col] = train[col].fillna(train[col].median())

Remplacer les NA valeurs catégorielles par "Unknown"

In [6]:
for col in train.select_dtypes(include=["object", "string"]).columns:
    train[col] = train[col].fillna("Unknown")

Fusionner les informations des magasins (store.csv) avec les ventes

In [7]:
train = train.merge(store, on="Store", how="left")
test = test.merge(store, on="Store", how="left")
train.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StateHoliday_flag,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,3,2013-01-02,5530,668,1,0,0,1,0,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,1,4,2013-01-03,4327,578,1,0,0,1,0,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
2,1,5,2013-01-04,4486,619,1,0,0,1,0,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
3,1,6,2013-01-05,4997,635,1,0,0,1,0,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
4,1,1,2013-01-07,7176,785,1,1,0,1,0,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN


Transformer la colonne Date en :
Année
Mois
Jour de la semaine


In [8]:
train["Date"] = pd.to_datetime(train["Date"])
train["Year"] = train["Date"].dt.year
train["Month"] = train["Date"].dt.month
train["DayOfWeek"] = train["Date"].dt.dayofweek
train[["Year", "Month", "DayOfWeek"]].head()

,Year,Month,DayOfWeek
0,2013,1,2
1,2013,1,3
2,2013,1,4
3,2013,1,5
4,2013,1,0


 Création de features d'interaction
.Combinaison des colonnes
.Utilisation des colonnes catégorielles pour capturer des effets combinés

In [9]:
# 💡 Création de features d'interaction
# 

interaction_pairs = [
    ("Promo", "StoreType"),
    ("Promo", "Assortment"),
    ("StoreType", "Assortment"),
    ("Month", "StoreType"),
    ("DayOfWeek", "StoreType"),
    ("Promo2_active", "Promo"),
]

def add_interactions(df):
    for a, b in interaction_pairs:
        colname = f"{a}_{b}"
        # si l'une des colonnes est absente on saute
        if a in df.columns and b in df.columns:
            df[colname] = df[a].astype(str) + "_" + df[b].astype(str)
    return df

# appliquer à tous les ensembles disponibles (train/test)
train = add_interactions(train)
if 'test' in globals():
    test = add_interactions(test)

# après le split, on ajoutera les interactions aux sous-ensembles

display(train.filter(like="Promo_StoreType").head())


,Promo_StoreType
0,0_c
1,0_c
2,0_c
3,0_c
4,1_c


Trier par magasin puis par date et réinitialiser les indexes

In [10]:
train = train.sort_values(["Store", "Date"])
train = train.reset_index(drop=True)
train.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StateHoliday_flag,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Year,Month,Promo_StoreType,Promo_Assortment,StoreType_Assortment,Month_StoreType,DayOfWeek_StoreType
0,1,2,2013-01-02,5530,668,1,0,0,1,0,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,2013,1,0_c,0_a,c_a,1_c,2_c
1,1,3,2013-01-03,4327,578,1,0,0,1,0,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,2013,1,0_c,0_a,c_a,1_c,3_c
2,1,4,2013-01-04,4486,619,1,0,0,1,0,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,2013,1,0_c,0_a,c_a,1_c,4_c
3,1,5,2013-01-05,4997,635,1,0,0,1,0,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,2013,1,0_c,0_a,c_a,1_c,5_c
4,1,0,2013-01-07,7176,785,1,1,0,1,0,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN,2013,1,1_c,1_a,c_a,1_c,0_c


Comparaison train vs test

In [11]:

def compare_distros(train_df, test_df):
    print("Open ratio train:", (train_df["Open"] == 1).mean())
    print("Open ratio test :", (test_df["Open"] == 1).mean())
    print("Promo ratio train:", train_df["Promo"].mean())
    print("Promo ratio test :", test_df["Promo"].mean())
    train_month = train_df["Date"].dt.month.value_counts(normalize=True).sort_index()
    test_month  = test_df["Date"].dt.month.value_counts(normalize=True).sort_index()
    comp = pd.DataFrame({"train_month": train_month, "test_month": test_month}).fillna(0)
    print("\nDistribution mensuelle (train vs test):")
    display(comp)

# calcul sur les ensembles complets (après merge store)
compare_distros(train, test)

# Si on veut inspecter uniquement les jours ouverts / période sélectionnée:
# compare_distros(train_set, test)


Open ratio train: 1.0
Open ratio test : 0.8540936526479751
Promo ratio train: 0.4463556064040704
Promo ratio test : 0.3958333333333333

Distribution mensuelle (train vs test):


,train_month,test_month
Date,,
1,0.102252,0.000000
2,0.095032,0.000000
3,0.101825,0.000000
4,0.096793,0.000000
5,0.094866,0.000000
6,0.097794,0.000000
7,0.101353,0.000000
8,0.064442,0.645833
9,0.061967,0.354167


Validation temporelle plus sophistiquée

In [12]:
# 🕒 Validation temporelle plus sophistiquée
# fonction utilitaire qui découpe les dernières `val_weeks` semaines comme validation

# ✅ CORRECTION : simuler la période Kaggle (août-sept) avec l'année précédente
def temporal_split(df):
    val_start = pd.Timestamp("2014-08-01")
    val_end   = pd.Timestamp("2014-09-17")
    
    train_part = df[(df["Date"] < val_start) | (df["Date"] > val_end)].copy()
    val_part   = df[(df["Date"] >= val_start) & (df["Date"] <= val_end)].copy()
    return train_part, val_part

train_set, val_set = temporal_split(train)
print("train :", train_set["Date"].min().date(), "→", train_set["Date"].max().date())
print("val   :", val_set["Date"].min().date(),   "→", val_set["Date"].max().date())

train : 2013-01-01 → 2015-07-31
val   : 2014-08-01 → 2014-09-17


In [13]:
# 🧩 Ajouter les interactions après le découpage temporel
# (les colonnes existant dans `train` sont copiées dans train_set/val_set)
train_set = add_interactions(train_set)
val_set   = add_interactions(val_set)

print("Interactions ajoutées à train_set et val_set")

Interactions ajoutées à train_set et val_set


In [14]:
# Vérification du split temporel
print(f"Train : {train_set['Date'].min().date()} → {train_set['Date'].max().date()}")
print(f"Val   : {val_set['Date'].min().date()} → {val_set['Date'].max().date()}")
print(f"Taille train : {len(train_set):,} | val : {len(val_set):,}")

Train : 2013-01-01 → 2015-07-31
Val   : 2014-08-01 → 2014-09-17
Taille train : 805,939 | val : 38,399


In [15]:
# --- Features historiques magasin (calculées sur train_set uniquement) ---
# Assurer Sales_log
if "Sales_log" not in train_set.columns:
    train_set["Sales_log"] = np.log1p(train_set["Sales"])
if "Sales_log" not in val_set.columns:
    val_set["Sales_log"] = np.log1p(val_set["Sales"])

store_mean = train_set.groupby("Store")["Sales_log"].mean()
global_mean = train_set["Sales_log"].mean()

train_set["Store_mean_log_sales"] = train_set["Store"].map(store_mean).fillna(global_mean)
val_set["Store_mean_log_sales"]   = val_set["Store"].map(store_mean).fillna(global_mean)
test["Store_mean_log_sales"]      = test["Store"].map(store_mean).fillna(global_mean)

store_dow_mean = train_set.groupby(["Store", "DayOfWeek"])["Sales_log"].mean()

train_set["Store_DOW_mean_log_sales"] = (
    pd.Series(train_set.set_index(["Store", "DayOfWeek"]).index.map(store_dow_mean), index=train_set.index)
    .fillna(train_set["Store_mean_log_sales"])
)
val_set["Store_DOW_mean_log_sales"] = (
    pd.Series(val_set.set_index(["Store", "DayOfWeek"]).index.map(store_dow_mean), index=val_set.index)
    .fillna(val_set["Store_mean_log_sales"])
)
test["Store_DOW_mean_log_sales"] = (
    pd.Series(test.set_index(["Store", "DayOfWeek"]).index.map(store_dow_mean), index=test.index)
    .fillna(test["Store_mean_log_sales"])
)

print("✅ Features historiques ajoutées:", ["Store_mean_log_sales", "Store_DOW_mean_log_sales"])

✅ Features historiques ajoutées: ['Store_mean_log_sales', 'Store_DOW_mean_log_sales']


Transformation de la cible : log1p(Sales)
l'EDA a montré une distribution très asymétrique des ventes.
on crée une colonne `Sales_log` pour l'entraînement du modèle.

In [16]:

for df in (train_set, val_set):
    df["Sales_log"] = np.log1p(df["Sales"])

# Vérifier la transformation
print("Before (train_set) :", train_set["Sales"].describe())
print("After  (train_set) :", train_set["Sales_log"].describe())


Before (train_set) : count    805939.000000
mean       6962.079190
std        3110.645473
min          46.000000
25%        4861.000000
50%        6373.000000
75%        8368.000000
max       41551.000000
Name: Sales, dtype: float64
After  (train_set) : count    805939.000000
mean          8.758420
std           0.425639
min           3.850148
25%           8.489205
50%           8.759982
75%           9.032290
max          10.634701
Name: Sales_log, dtype: float64


Détection et traitement des outliers sur Sales
On regarde les quantiles et on plafonne (capping) entre 1er et 99e percentile.


In [17]:
q_low = train_set["Sales"].quantile(0.01)
q_high = train_set["Sales"].quantile(0.99)
print("Sales quantiles (1%, 99%):", q_low, q_high)

# ajout d'une colonne plafonnée
train_set["Sales_clipped"] = train_set["Sales"].clip(lower=q_low, upper=q_high)
val_set["Sales_clipped"]   = val_set["Sales"].clip(lower=q_low, upper=q_high)

# on peut aussi standardiser la cible log1p
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
train_set["Sales_log_scaled"] = scaler.fit_transform(train_set[["Sales_log"]])
val_set["Sales_log_scaled"]   = scaler.transform(val_set[["Sales_log"]])

print(train_set[["Sales","Sales_clipped","Sales_log","Sales_log_scaled"]].describe())


Sales quantiles (1%, 99%): 2188.0 17828.0
               Sales  Sales_clipped      Sales_log  Sales_log_scaled
count  805939.000000  805939.000000  805939.000000      8.059390e+05
mean     6962.079190    6936.003436       8.758420     -9.298764e-16
std      3110.645473    2966.015355       0.425639      1.000001e+00
min        46.000000    2188.000000       3.850148     -1.153154e+01
25%      4861.000000    4861.000000       8.489205     -6.324962e-01
50%      6373.000000    6373.000000       8.759982      3.670392e-03
75%      8368.000000    8368.000000       9.032290      6.434313e-01
max     41551.000000   17828.000000      10.634701      4.408150e+00


Ne pas utiliser Customers comme feature du modèle,Car on connais pas les nombres des clients du futur

In [18]:
train = train.drop(columns=["Customers"], errors="ignore")
test = test.drop(columns=["Customers"], errors="ignore")

Promo2 active 

In [19]:
# Mapping mois numéro -> abréviation
month_map = {
    1: "Jan", 2: "Feb", 3: "Mar", 4: "Apr",
    5: "May", 6: "Jun", 7: "Jul", 8: "Aug",
    9: "Sept", 10: "Oct", 11: "Nov", 12: "Dec"
}

def add_promo2_active(df):
    # créer colonne par défaut
    df["Promo2_active"] = 0

    # vérifier colonnes nécessaires
    if "Promo2" not in df.columns or "PromoInterval" not in df.columns:
        return df

    # garantir la colonne Month (si absente, la dériver depuis Date)
    if "Month" not in df.columns and "Date" in df.columns:
        df["Month"] = df["Date"].dt.month

    if "Month" not in df.columns:
        return df

    # mapper month -> abréviation
    months = df["Month"].map(month_map)
    intervals = df["PromoInterval"].fillna("").astype(str)

    # traiter seulement les lignes avec Promo2 == 1 (réduit le travail Python)
    mask = df["Promo2"] == 1
    if mask.any():
        # construction en mémoire via zip (plus rapide que df.loc dans une boucle)
        vals = [1 if mn in iv else 0 for mn, iv in zip(months[mask], intervals[mask])]
        df.loc[mask, "Promo2_active"] = vals

    return df

# Appliquer sur train et validation
train = add_promo2_active(train)
val_set = add_promo2_active(val_set)

In [20]:
train.loc[train["Promo2_active"]==1, ["Promo2","PromoInterval","Month"]]

,Promo2,PromoInterval,Month
781,1,"Jan,Apr,Jul,Oct",1
782,1,"Jan,Apr,Jul,Oct",1
783,1,"Jan,Apr,Jul,Oct",1
784,1,"Jan,Apr,Jul,Oct",1
785,1,"Jan,Apr,Jul,Oct",1
...,...,...,...
844306,1,"Mar,Jun,Sept,Dec",6
844307,1,"Mar,Jun,Sept,Dec",6
844308,1,"Mar,Jun,Sept,Dec",6
844309,1,"Mar,Jun,Sept,Dec",6


L'Age du concurrent

In [21]:
# 1️⃣ Créer une date d'ouverture du concurrent
train["CompetitionOpenDate"] = pd.to_datetime(
    dict(year=train["CompetitionOpenSinceYear"],
         month=train["CompetitionOpenSinceMonth"],
         day=1),
    errors="coerce"
)

# 2️⃣ Calculer l'âge de la concurrence en mois (différence année*12 + mois)
train["CompetitionAge"] = (
    (train["Date"].dt.year - train["CompetitionOpenDate"].dt.year) * 12 +
    (train["Date"].dt.month - train["CompetitionOpenDate"].dt.month)
)

# 3️⃣ Remplacer valeurs négatives ou NA
train["CompetitionAge"] = train["CompetitionAge"].clip(lower=0)
train["CompetitionAge"] = train["CompetitionAge"].fillna(0)

Traiter correctement les valeurs manquantes des variables du store :

En créant un flag NA

In [22]:
store_na_cols = [
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
    "Promo2SinceWeek",
    "Promo2SinceYear"
]

# Garantir existence des colonnes dans train/test
for col in store_na_cols:
    if col not in train.columns:
        train[col] = np.nan
    if col not in test.columns:
        test[col] = np.nan

# Créer indicateurs de missing et remplir avec la médiane calculée sur le train
for col in store_na_cols:
    train[f"{col}_missing"] = train[col].isna().astype(int)
    test[f"{col}_missing"]  = test[col].isna().astype(int)

    median_value = train[col].median()
    train[col] = train[col].fillna(median_value)
    test[col]  = test[col].fillna(median_value)

In [23]:
# ═══════════════════════════════════════════════════════════
# 📊 FEATURES D'AGRÉGATION PAR MAGASIN
# Calculées sur train_set uniquement → 0% de NaN, pas de fuite
# ═══════════════════════════════════════════════════════════

global_mean = train_set["Sales_log"].mean()

# Moyenne globale par magasin
store_mean = train_set.groupby("Store")["Sales_log"].mean()

# Moyenne par (magasin, mois) → capture la saisonnalité propre à chaque magasin
store_month_mean = train_set.groupby(["Store", "Month"])["Sales_log"].mean()

# Moyenne par (magasin, jour de semaine)
store_dow_mean = train_set.groupby(["Store", "DayOfWeek"])["Sales_log"].mean()

# Moyenne par (magasin, promo) → l'effet promo varie selon le magasin
store_promo_mean = train_set.groupby(["Store", "Promo"])["Sales_log"].mean()

# Moyenne par (magasin, mois, promo) → le plus précis
store_month_promo_mean = train_set.groupby(["Store", "Month", "Promo"])["Sales_log"].mean()

# Médiane et std par magasin (robustesse aux outliers)
store_median  = train_set.groupby("Store")["Sales_log"].median()
store_std     = train_set.groupby("Store")["Sales_log"].std()

# Médiane par (magasin, mois) → plus robuste que la moyenne
store_month_median = train_set.groupby(["Store", "Month"])["Sales_log"].median()

def add_store_agg_features(df):
    df = df.copy()
    # S'assurer que Month et DayOfWeek existent (test ne les a pas encore)
    if "Month" not in df.columns:
        df["Month"] = pd.to_datetime(df["Date"]).dt.month
    if "DayOfWeek" not in df.columns:
        df["DayOfWeek"] = pd.to_datetime(df["Date"]).dt.dayofweek + 1

    df["Store_mean_sales"]       = df["Store"].map(store_mean).fillna(global_mean)
    df["Store_Month_mean_sales"] = df.set_index(["Store", "Month"]).index.map(store_month_mean).values
    df["Store_DOW_mean_sales"]   = df.set_index(["Store", "DayOfWeek"]).index.map(store_dow_mean).values
    df["Store_Promo_mean_sales"] = df.set_index(["Store", "Promo"]).index.map(store_promo_mean).values
    df["Store_Month_Promo_mean"] = df.set_index(["Store", "Month", "Promo"]).index.map(store_month_promo_mean).values
    df["Store_median_sales"]       = df["Store"].map(store_median).fillna(global_mean)
    df["Store_std_sales"]          = df["Store"].map(store_std).fillna(0)
    df["Store_Month_median_sales"] = df.set_index(["Store","Month"]).index.map(store_month_median).values
    df["Store_Month_median_sales"] = df["Store_Month_median_sales"].fillna(df["Store_median_sales"])
    for col in ["Store_Month_mean_sales", "Store_DOW_mean_sales",
                "Store_Promo_mean_sales", "Store_Month_Promo_mean"]:
        df[col] = df[col].fillna(df["Store_mean_sales"])
    return df

train_set = add_store_agg_features(train_set)
val_set   = add_store_agg_features(val_set)
test      = add_store_agg_features(test)

# Vérification
nan_cols = ["Store_mean_sales", "Store_Month_mean_sales", 
            "Store_DOW_mean_sales", "Store_Promo_mean_sales", "Store_Month_Promo_mean"]
print("✅ Features d'agrégation créées — NaN :")
for col in nan_cols:
    print(f"  {col}: train={train_set[col].isna().sum()} | val={val_set[col].isna().sum()} | test={test[col].isna().sum()}")

✅ Features d'agrégation créées — NaN :
  Store_mean_sales: train=0 | val=0 | test=0
  Store_Month_mean_sales: train=0 | val=0 | test=0
  Store_DOW_mean_sales: train=0 | val=0 | test=0
  Store_Promo_mean_sales: train=0 | val=0 | test=0
  Store_Month_Promo_mean: train=0 | val=0 | test=0


##  plan d’expérience

In [ ]:
# 🎯 Définir la liste des features à utiliser
# On exclut certaines colonnes qui ne doivent pas être des features

exclude_cols = [
    "Sales",
    "Sales_log", "Sales_clipped",
    "Sales_log_scaled",
    "Date",
    "Open",
    "Customers",
    "PromoInterval",
    "CompetitionOpenDate",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear",
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "Year",           # ← AJOUTE CETTE LIGNE
]

# lister les features restantes (excluant les NA-flags pour l'instant)
feature_cols = [col for col in train.columns if col not in exclude_cols]
print(f"Total features avant dummies: {len(feature_cols)}")
print("Features sélectionnés:", feature_cols)

# Ajouter manuellement les lag features (elles sont dans train_set, pas dans train)
lag_feature_names = ["Sales_lag_1y", "Sales_lag_1y_week"]
feature_cols = feature_cols + [f for f in lag_feature_names if f in train_set.columns]
print(f"Features après ajout des lags : {len(feature_cols)} (dont {len(lag_feature_names)} lag features)")

agg_feature_names = [
    "Store_mean_sales", "Store_Month_mean_sales",
    "Store_DOW_mean_sales", "Store_Promo_mean_sales", "Store_Month_Promo_mean",
    "Store_median_sales", "Store_std_sales", "Store_Month_median_sales"
]
feature_cols = feature_cols + [f for f in agg_feature_names if f in train_set.columns]
print(f"Features après ajout des agrégations : {len(feature_cols)} (dont {len(agg_feature_names)} agg features)")


Total features avant dummies: 24
Features sélectionnés: ['Store', 'DayOfWeek', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StateHoliday_flag', 'StoreType', 'Assortment', 'CompetitionDistance', 'Promo2', 'Year', 'Month', 'Promo_StoreType', 'Promo_Assortment', 'StoreType_Assortment', 'Month_StoreType', 'DayOfWeek_StoreType', 'Promo2_active', 'CompetitionAge', 'CompetitionDistance_missing', 'CompetitionOpenSinceMonth_missing', 'CompetitionOpenSinceYear_missing', 'Promo2SinceWeek_missing', 'Promo2SinceYear_missing']
Features après ajout des lags : 24 (dont 2 lag features)
Features après ajout des agrégations : 32 (dont 8 agg features)


### Synchroniser test avec les colonnes créées dans train
### Créer les features manquantes sur test

In [25]:


# 1 Year, Month, DayOfWeek (de Date)
if 'Year' not in test.columns:
    test['Year'] = test['Date'].dt.year
if 'Month' not in test.columns:
    test['Month'] = test['Date'].dt.month
if 'DayOfWeek' not in test.columns:
    # DayOfWeek Rossmann: 1=Mon ... 7=Sun
    test['DayOfWeek'] = test['Date'].dt.dayofweek + 1

# 2 StateHoliday encoding (s'il n'existe pas)
if 'StateHoliday_flag' not in test.columns:
    test['StateHoliday_flag'] = (test['StateHoliday'] != '0').astype(int)
    test['StateHoliday_type'] = test['StateHoliday'].map({'0': 0, 'a': 1, 'b': 2, 'c': 3}).fillna(0)

# 3 Promo2_active
if 'Promo2_active' not in test.columns:
    month_names = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun', 
                   7: 'Jul', 8: 'Aug', 9: 'Sept', 10: 'Oct', 11: 'Nov', 12: 'Dec'}
    
    # Wrap each condition in (brackets) and use & for row-by-row logic
    test['Promo2_active'] = (
        (test['Promo2'].astype(bool)) & 
        (test['PromoInterval'].isna() == False) & 
        (test.apply(lambda x: month_names[x['Month']] in str(x['PromoInterval']), axis=1))
    ).astype(int)

# 4 CompetitionAge
if 'CompetitionAge' not in test.columns:
    def calc_age(row):
        if pd.isna(row['CompetitionOpenSinceYear']) or row['CompetitionOpenSinceYear'] == 0:
            return np.nan
        year_diff = row['Year'] - row['CompetitionOpenSinceYear']
        month_diff = row['Month'] - (row['CompetitionOpenSinceMonth'] if not pd.isna(row['CompetitionOpenSinceMonth']) else 1)
        return year_diff * 12 + month_diff
    test['CompetitionAge'] = test.apply(calc_age, axis=1)

# 5 Missing indicators
missing_cols = ['CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear', 'Promo2SinceWeek', 'Promo2SinceYear']
for col in missing_cols:
    if f'{col}_missing' not in test.columns:
        test[f'{col}_missing'] = test[col].isna().astype(int)

# 6 Interactions
if 'Promo_StoreType' not in test.columns:
    test['Promo_StoreType'] = test['Promo'].astype(str) + '_' + test['StoreType'].astype(str)
if 'Promo_Assortment' not in test.columns:
    test['Promo_Assortment'] = test['Promo'].astype(str) + '_' + test['Assortment'].astype(str)
if 'StoreType_Assortment' not in test.columns:
    test['StoreType_Assortment'] = test['StoreType'].astype(str) + '_' + test['Assortment'].astype(str)
if 'Month_StoreType' not in test.columns:
    test['Month_StoreType'] = test['Month'].astype(str) + '_' + test['StoreType'].astype(str)
if 'DayOfWeek_StoreType' not in test.columns:
    test['DayOfWeek_StoreType'] = test['DayOfWeek'].astype(str) + '_' + test['StoreType'].astype(str)
if 'Promo2_Promo' not in test.columns:
    test['Promo2_Promo'] = test['Promo2_active'].astype(str) + '_' + test['Promo'].astype(str)

print(f"Features créées sur test\n  test columns: {len(test.columns)}\n  test shape: {test.shape}")

# Vérifier que tous les features existent à la fois dans test et train_set
missing_in_test = set(feature_cols) - set(test.columns)
if missing_in_test:
    print(f" Colonnes manquantes dans test: {missing_in_test}")
else:
    print(f" Toutes les colonnes requises existent dans test")

Features créées sur test
  test columns: 43
  test shape: (41088, 43)
 Toutes les colonnes requises existent dans test


In [26]:
# ⚠️ Synchroniser train_set et val_set avec les colonnes crées après le split
# Les colonnes comme Promo2_active, CompetitionAge ont été ajoutées à 'train' après
# le split initial, donc il faut les copier dans train_set et val_set

train_set = train_set.copy()
val_set = val_set.copy()

# Ajouter les colonnes crées après le split
cols_to_sync = [
    "Promo2_active", "CompetitionAge", "Sales_log", "Sales_clipped", "Sales_log_scaled",
    "CompetitionDistance_missing", "CompetitionOpenSinceMonth_missing", 
    "CompetitionOpenSinceYear_missing", "Promo2SinceWeek_missing", "Promo2SinceYear_missing",
    "StateHoliday_flag", "StateHoliday_type"
]

for col in cols_to_sync:
    if col in train.columns and col not in train_set.columns:
        # localiser les lignes de train_set dans train en utilisant Store/Date
        train_set[col] = train.loc[train_set.index, col].values
    if col in train.columns and col not in val_set.columns:
        val_set[col] = train.loc[val_set.index, col].values

print("Colonnes synchronisées dans train_set et val_set")
print(f"  train_set columns: {len(train_set.columns)}")
print(f"  val_set columns:   {len(val_set.columns)}")


Colonnes synchronisées dans train_set et val_set
  train_set columns: 46
  val_set columns:   46


In [27]:
# 🔢 Encoder les catégories en variables indicatrices (one-hot encoding)

# Identifier colonnes catégorielles (sur le train)
cat_cols = train_set[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Colonnes catégorielles: {cat_cols}")

# ✅ Recommandé pour Kaggle/arbres : drop_first=False (stabilité + pas d'ambiguïté de catégorie de référence)
train_set_encoded = pd.get_dummies(train_set[feature_cols], columns=cat_cols, drop_first=False)
val_set_encoded   = pd.get_dummies(val_set[feature_cols],   columns=cat_cols, drop_first=False)
test_encoded      = pd.get_dummies(test[feature_cols],      columns=cat_cols, drop_first=False)

print(f"\nShape après dummies:")
print(f"  train_set: {train_set_encoded.shape}")
print(f"  val_set:   {val_set_encoded.shape}")
print(f"  test:      {test_encoded.shape}")

# Aligner les colonnes (val/test peuvent avoir colonnes manquantes)
missing_cols_val  = set(train_set_encoded.columns) - set(val_set_encoded.columns)
missing_cols_test = set(train_set_encoded.columns) - set(test_encoded.columns)

for col in missing_cols_val:
    val_set_encoded[col] = 0
for col in missing_cols_test:
    test_encoded[col] = 0

# Supprimer colonnes en trop dans val/test
val_set_encoded = val_set_encoded[train_set_encoded.columns]
test_encoded    = test_encoded[train_set_encoded.columns]

print(f"\nShape après alignement:")
print(f"  train_set: {train_set_encoded.shape}")
print(f"  val_set:   {val_set_encoded.shape}")
print(f"  test:      {test_encoded.shape}")
print(f"Column list identical: {train_set_encoded.columns.equals(val_set_encoded.columns) and train_set_encoded.columns.equals(test_encoded.columns)}")


Colonnes catégorielles: ['StateHoliday', 'StoreType', 'Assortment', 'Promo_StoreType', 'Promo_Assortment', 'StoreType_Assortment', 'Month_StoreType', 'DayOfWeek_StoreType']


C:\Users\66kalina\AppData\Local\Temp\ipykernel_24792\3006533794.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train_set[feature_cols].select_dtypes(include=['object', 'category']).columns.tolist()



Shape après dummies:
  train_set: (805939, 133)
  val_set:   (38399, 90)
  test:      (41088, 92)

Shape après alignement:
  train_set: (805939, 133)
  val_set:   (38399, 133)
  test:      (41088, 133)
Column list identical: True


In [28]:
# 📊 Construire les matrices X (features) et y (cible)
# Pour train_set et val_set (test n'a pas de cible)

# Cible : on peut utiliser Sales_log ou Sales_log_scaled selon le modèle
target_col = "Sales_log"  # utiliser log1p pour stabiliser la variance

X_train = train_set_encoded.copy()
y_train = train_set[target_col].copy()

X_val = val_set_encoded.copy()
y_val = val_set[target_col].copy()

X_test = test_encoded.copy()

print(f"Dimensions finales:")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_val:   {X_val.shape},   y_val:   {y_val.shape}")
print(f"  X_test:  {X_test.shape}")

print(f"\nSample du y_train (premiers 5 éléments):")
print(y_train.head())
print(f"\nColonnes de X_train (premiers 10):")
print(X_train.columns[:10].tolist())


Dimensions finales:
  X_train: (805939, 133), y_train: (805939,)
  X_val:   (38399, 133),   y_val:   (38399,)
  X_test:  (41088, 133)

Sample du y_train (premiers 5 éléments):
0    8.618124
1    8.372861
2    8.408940
3    8.516793
4    8.878637
Name: Sales_log, dtype: float64

Colonnes de X_train (premiers 10):
['Store', 'DayOfWeek', 'Promo', 'SchoolHoliday', 'StateHoliday_flag', 'CompetitionDistance', 'Promo2', 'Year', 'Month', 'Promo2_active']


In [29]:
# 💾 Sauvegarder les données préparées pour la modélisation
# Format : CSV ou Pickle pour réutiliser rapidement dans le notebook 03

import pickle

# créer un dossier de sortie si nécessaire
output_dir = BASE_DIR / "data" / "processed"
output_dir.mkdir(exist_ok=True)

# Sauvegarder en CSV (format lisible)
X_train.to_csv(output_dir / "X_train.csv", index=False)
y_train.to_csv(output_dir / "y_train.csv", index=False)
X_val.to_csv(output_dir / "X_val.csv", index=False)
y_val.to_csv(output_dir / "y_val.csv", index=False)
X_test.to_csv(output_dir / "X_test.csv", index=False)

# ✅ Sauvegarder Id/Open dans le même ordre que X_test (évite mismatch à la soumission)
test_meta = test[["Id", "Open"]].copy()
test_meta.to_csv(output_dir / "test_meta.csv", index=False)
print(f"  {output_dir / 'test_meta.csv'}")

print("✅ Fichiers CSV sauvegardés:")
print(f"  {output_dir / 'X_train.csv'}")
print(f"  {output_dir / 'y_train.csv'}")
print(f"  {output_dir / 'X_val.csv'}")
print(f"  {output_dir / 'y_val.csv'}")
print(f"  {output_dir / 'X_test.csv'}")

# Sauvegarder aussi la liste des colonnes pour traçabilité
col_mapping = {
    "feature_cols": feature_cols,
    "cat_cols": cat_cols,
    "exclude_cols": exclude_cols,
    "target_col": target_col,
    "X_columns": X_train.columns.tolist()
}
with open(output_dir / "column_mapping.pkl", "wb") as f:
    pickle.dump(col_mapping, f)

print("✅ Métadonnées (colonnes, target) sauvegardées")

print(f"\nPrêt pour modélisation ! Les données encodées sont dans: {output_dir}")


  e:\Code\rossmann-kaggle\data\processed\test_meta.csv
✅ Fichiers CSV sauvegardés:
  e:\Code\rossmann-kaggle\data\processed\X_train.csv
  e:\Code\rossmann-kaggle\data\processed\y_train.csv
  e:\Code\rossmann-kaggle\data\processed\X_val.csv
  e:\Code\rossmann-kaggle\data\processed\y_val.csv
  e:\Code\rossmann-kaggle\data\processed\X_test.csv
✅ Métadonnées (colonnes, target) sauvegardées

Prêt pour modélisation ! Les données encodées sont dans: e:\Code\rossmann-kaggle\data\processed
